In [62]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split

In [63]:
df = pd.read_csv("../../data/processed/laptop_train_processed.csv")
df.head()

,aspect_context,label
0,night and skip taking the [ASPECT] with me bec...,1
1,me because of the good [ASPECT],2
2,tech guy then said the [ASPECT] does not do 1-...,0
3,direct my concern to the [ASPECT] which is the...,0
4,the [ASPECT] then said the service center,1


In [64]:
# only keep model inputs

X = df['aspect_context']
y = df['label']

In [65]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [66]:
# it is preprocessed in such a way that each word is separated by single-space

def tokenize(text):
    return text.split()

In [67]:
train_tokens = X_train.apply(tokenize)
train_tokens.head()

1495    [and, the, screen, on, this, thing, is, absolu...
1551    [they, were, shipping, out, bad, [ASPECT], and...
1829    [another, [ASPECT], that, is, laughable, is, the]
1269    [as, a, die-hard, [ASPECT], enthusiast,, i, sh...
1277    [any, laptop, not, purchased, with, [ASPECT], ...
Name: aspect_context, dtype: object

In [68]:
# collect only unique tokens in training set

vocab = set()

for tokens in train_tokens:
    vocab.update(tokens)

vocab = sorted(vocab)
len(vocab)

2959

# Computing TF-IDF for each tokens

In [69]:
train_tokens.iloc[0]

['and',
 'the',
 'screen',
 'on',
 'this',
 'thing',
 'is',
 'absolutely',
 'amazing',
 'for',
 'high',
 'quality',
 'videos',
 'and',
 'movies',
 'and',
 'gaming.']

In [70]:
# this computes document frequency for each word in the vocabulary
# number of documents in which the word appears


doc_freq = {}

for word in vocab:
    doc_freq[word] = sum(word in tokens for tokens in train_tokens)

In [71]:
# compute IDF for each word


N = len(train_tokens)

idf = {
    word: np.log(N / (1 + doc_freq[word]))
    for word in vocab
}

In [72]:
# TF-IDF computation

# below is how it works
# compute TF-IDF vector for a given token list
# [tf-idf(w) = tf(w) * idf(w) for each word w in vocab]



def compute_tfidf(tokens, vocab, idf):
    tfidf_vector = np.zeros(len(vocab))
    token_counts = Counter(tokens)
    total_tokens = len(tokens)
    
    for i, word in enumerate(vocab):
        tf = token_counts[word] / total_tokens if total_tokens > 0 else 0
        tfidf_vector[i] = tf * idf[word]
    
    return tfidf_vector

In [73]:
# create TF-IDF matrix for training set

X_train_tfidf = np.array([compute_tfidf(tokens, vocab, idf) for tokens in train_tokens])

X_train_tfidf.shape

(1850, 2959)

In [74]:
# transform test set

test_tokens = X_test.apply(tokenize)

X_test_tfidf = np.array([compute_tfidf(tokens, vocab, idf) for tokens in test_tokens])
X_test_tfidf.shape

(463, 2959)

In [75]:
X_train_tfidf

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(1850, 2959))

In [76]:
# checking once if it is


[i for i in zip(
    np.where(X_train_tfidf[0] != 0)[0][:10],
    [vocab[i] for i in np.where(X_train_tfidf[0] != 0)[0][:10]]
)]

[(np.int64(192), 'absolutely'),
 (np.int64(262), 'amazing'),
 (np.int64(273), 'and'),
 (np.int64(1058), 'for'),
 (np.int64(1108), 'gaming.'),
 (np.int64(1236), 'high'),
 (np.int64(1381), 'is'),
 (np.int64(1689), 'movies'),
 (np.int64(1804), 'on'),
 (np.int64(2067), 'quality')]

# Manual implementation of MNB

In [77]:
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [78]:
# print(X_train_tfidf.shape)
# print(X_test_tfidf.shape)
# print(y_train.shape)
# print(y_test.shape)


In [79]:
classes = np.unique(y_train)
classes

array([0, 1, 2])

In [80]:
# compute class priors P(C) = documents in class C / total documents

class_priors = {}

for c in classes:
    class_priors[c] = np.sum(y_train == c) / len(y_train)

class_priors

{np.int64(0): np.float64(0.3745945945945946),
 np.int64(1): np.float64(0.1989189189189189),
 np.int64(2): np.float64(0.42648648648648646)}

In [81]:
# compute likelihoods P(X|C) with Laplace smoothing
# P(X|C) = (count(X in C) + alpha) / (total count of words in C + alpha*num_features)

alpha = 1.0 # Laplace smoothing parameter
num_features = X_train_tfidf.shape[1]


feature_likelihoods = {}

for c in classes:
    X_c = X_train_tfidf[y_train == c]      
    feature_sum = X_c.sum(axis=0)

    feature_likelihoods[c] = ((feature_sum + alpha) / (feature_sum.sum() + alpha*num_features))


In [82]:
# make predictions
# arg y_max​[logP(y) + ∑​xi​ log(P(wi​∣y))]


def predict_naive_bayes(X, class_priors, feature_likelihoods):
    predictions = []
    
    for x in X:
        scores = {}
        
        for c in class_priors:
            log_prior = np.log(class_priors[c])
            log_likelihood = np.sum(x * np.log(feature_likelihoods[c]))
            scores[c] = log_prior + log_likelihood
        
        predictions.append(max(scores, key=scores.get))
    
    return np.array(predictions)


In [83]:
y_pred_manual = predict_naive_bayes(
    X_test_tfidf,
    class_priors,
    feature_likelihoods
)

y_pred_manual[:10]

array([0, 0, 2, 0, 2, 2, 0, 2, 2, 2])

In [84]:
accuracy = accuracy_score(y_test, y_pred_manual)
print("Accuracy: {:.2f}%".format(accuracy*100))

Accuracy: 70.19%


In [85]:
print(classification_report(
    y_test,
    y_pred_manual,
    target_names=['negative', 'neutral', 'positive']
))

              precision    recall  f1-score   support

    negative       0.67      0.79      0.73       173
     neutral       0.75      0.23      0.35        92
    positive       0.72      0.84      0.78       198

    accuracy                           0.70       463
   macro avg       0.71      0.62      0.62       463
weighted avg       0.71      0.70      0.67       463



In [86]:
confusion_matrix(y_test, y_pred_manual)

array([[137,   2,  34],
       [ 41,  21,  30],
       [ 26,   5, 167]])

In [87]:
results_df = pd.DataFrame({
    'aspect_context': X_test.values[:10],
    'true_label': y_test.values[:10],
    'predicted_label': y_pred_manual[:10]
})

results_df

,aspect_context,true_label,predicted_label
0,netbook was desinged for was [ASPECT] and boy ...,2,0
1,"either way, the [ASPECT] is unsatisfactory.",0,0
2,"great os, fabulous improvements to the existin...",2,2
3,didn't [ASPECT] when shipped from walmart.com but,0,0
4,the [ASPECT] is like no other laptop,1,2
5,the [ASPECT] is incredible and i am,2,2
6,"all dark, power light steady, [ASPECT] steady ...",0,0
7,"to do simple things, like [ASPECT]",0,2
8,the difference is it's a whole lot of fun usin...,2,2
9,"screen, keyboard, and mouse: if you cant see y...",1,2


In [88]:
label_decode = {
    0: 'negative',
    1: 'neutral',
    2: 'positive'
}

results_df['true_label'] = results_df['true_label'].map(label_decode)
results_df['predicted_label'] = results_df['predicted_label'].map(label_decode)

results_df

,aspect_context,true_label,predicted_label
0,netbook was desinged for was [ASPECT] and boy ...,positive,negative
1,"either way, the [ASPECT] is unsatisfactory.",negative,negative
2,"great os, fabulous improvements to the existin...",positive,positive
3,didn't [ASPECT] when shipped from walmart.com but,negative,negative
4,the [ASPECT] is like no other laptop,neutral,positive
5,the [ASPECT] is incredible and i am,positive,positive
6,"all dark, power light steady, [ASPECT] steady ...",negative,negative
7,"to do simple things, like [ASPECT]",negative,positive
8,the difference is it's a whole lot of fun usin...,positive,positive
9,"screen, keyboard, and mouse: if you cant see y...",neutral,positive
